In [1]:
import CAST
import scanpy as sc
import os
import numpy as np
import warnings
import dgl
import torch
import CAST
import os
import numpy as np
import anndata as ad
import scanpy as sc
import warnings
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from sklearn.linear_model import LinearRegression
warnings.filterwarnings("ignore")

同cluster内找点与点间对应关系

In [ ]:
grp='dn3'
out_dir =f'/p2/zulab/jtian/data/SA/06_calculateConcentration/output_Banksy_PixelLR/{grp}'

In [ ]:

# 4 个 pair 文件路径
pair_files = {
    "15vs0":  f"/p2/zulab/jtian/data/SA/05_CAST/{grp}/output_cast1/15_align_to_0_coords_final.data",
    "30vs15": f"/p2/zulab/jtian/data/SA/05_CAST/{grp}/output_cast1/30_align_to_15_coords_final.data",
    "45vs30": f"/p2/zulab/jtian/data/SA/05_CAST/{grp}/output_cast1/45_align_to_30_coords_final.data",
    "60vs45": f"/p2/zulab/jtian/data/SA/05_CAST/{grp}/output_cast1/60_align_to_45_coords_final.data",
}

# 你想要的 8 组列（每组会扁平化成 _x / _y 两列）
# (pair_name, sample_id_in_that_file, base_colname)
col_plan = [
    ("15vs0",  "0",  "0-15vs0"),
    ("15vs0",  "15", "15-15vs0"),

    ("30vs15", "15", "15-30vs15"),
    ("30vs15", "30", "30-30vs15"),

    ("45vs30", "30", "30-45vs30"),
    ("45vs30", "45", "45-45vs30"),

    ("60vs45", "45", "45-60vs45"),
    ("60vs45", "60", "60-60vs45"),
]

def to_numpy_xy(v):
    if torch.is_tensor(v):
        v = v.detach().cpu().numpy()
    else:
        v = np.asarray(v)
    return v  # 期望 (N,2)

# 1) 读入四个 dict（key 统一转 str）
loaded = {}
for pair_name, fp in pair_files.items():
    d = torch.load(fp, map_location="cpu")
    loaded[pair_name] = {str(k): v for k, v in d.items()}

# 2) 按顺序直接拼列（不做任何合并/覆盖；不做 KeyError 检查）
dfs = []
for pair_name, sample_id, base_col in col_plan:
    xy = to_numpy_xy(loaded[pair_name][str(sample_id)])
    df_xy = pd.DataFrame(xy, columns=[f"{base_col}_x", f"{base_col}_y"])
    dfs.append(df_xy)

merged = pd.concat(dfs, axis=1)

# 3) 保存

os.makedirs(out_dir, exist_ok=True)
out_csv = os.path.join(out_dir, "aligned_coords.csv")
merged.to_csv(out_csv, index=False)

print("Saved:", out_csv)
print("Shape:", merged.shape)  # 预期列数 = 16
display(merged.head())


Saved: /p2/zulab/jtian/data/SA/06_calculateConcentration/output_Banksy_PixelLR/dn3/aligned_coords.csv
Shape: (63282, 16)


,0-15vs0_x,0-15vs0_y,15-15vs0_x,15-15vs0_y,15-30vs15_x,15-30vs15_y,30-30vs15_x,30-30vs15_y,30-45vs30_x,30-45vs30_y,45-45vs30_x,45-45vs30_y,45-60vs45_x,45-60vs45_y,60-60vs45_x,60-60vs45_y
0,56099.0,6544.0,55889.605469,87.594566,43279.0,11864.0,43218.527344,11797.666992,29659.0,11404.0,29600.683594,11409.931641,15479.0,11404.0,15451.801758,11458.494141
1,56119.0,6544.0,55869.636719,89.182701,43299.0,11864.0,43238.296875,11798.726562,29679.0,11404.0,29620.697266,11409.502930,15499.0,11404.0,15472.122070,11459.142578
2,55959.0,6524.0,55849.664062,90.771080,43319.0,11864.0,43258.062500,11799.787109,29699.0,11404.0,29640.712891,11409.074219,15519.0,11404.0,15492.442383,11459.791016
3,55979.0,6524.0,55951.171875,103.617516,43219.0,11844.0,43277.832031,11800.847656,29719.0,11404.0,29660.726562,11408.645508,15539.0,11404.0,15512.762695,11460.439453
4,55999.0,6524.0,55931.199219,105.205650,43239.0,11844.0,43297.601562,11801.907227,29739.0,11404.0,29680.740234,11408.216797,15559.0,11404.0,15533.083008,11461.087891


In [15]:
adata = ad.read_h5ad(f"/p2/zulab/jtian/data/SA/05_CAST/{grp}/output_cast1/demo1_cast1.h5ad") ###########
mask = adata.obs['sample']=='60'
c = adata.obs.loc[mask,['x','y']].copy()
coords = pd.concat([merged,c.reset_index(drop=True)],axis=1)

In [16]:
coords

,0-15vs0_x,0-15vs0_y,15-15vs0_x,15-15vs0_y,15-30vs15_x,15-30vs15_y,30-30vs15_x,30-30vs15_y,30-45vs30_x,30-45vs30_y,45-45vs30_x,45-45vs30_y,45-60vs45_x,45-60vs45_y,60-60vs45_x,60-60vs45_y,x,y
0,56099.0,6544.0,55889.605469,87.594566,43279.0,11864.0,43218.527344,11797.666992,29659.0,11404.0,29600.683594,11409.931641,15479.0,11404.0,15451.801758,11458.494141,2279.0,12064.0
1,56119.0,6544.0,55869.636719,89.182701,43299.0,11864.0,43238.296875,11798.726562,29679.0,11404.0,29620.697266,11409.502930,15499.0,11404.0,15472.122070,11459.142578,2299.0,12064.0
2,55959.0,6524.0,55849.664062,90.771080,43319.0,11864.0,43258.062500,11799.787109,29699.0,11404.0,29640.712891,11409.074219,15519.0,11404.0,15492.442383,11459.791016,2319.0,12064.0
3,55979.0,6524.0,55951.171875,103.617516,43219.0,11844.0,43277.832031,11800.847656,29719.0,11404.0,29660.726562,11408.645508,15539.0,11404.0,15512.762695,11460.439453,2339.0,12064.0
4,55999.0,6524.0,55931.199219,105.205650,43239.0,11844.0,43297.601562,11801.907227,29739.0,11404.0,29680.740234,11408.216797,15559.0,11404.0,15533.083008,11461.087891,2359.0,12064.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63277,56099.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63278,56119.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63279,56139.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63280,56159.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
coords.columns = [
    '0-original-x','0-original-y',
    '15-changed-x','15-changed-y',
    '15-original-x','15-original-y',
    '30-changed-x','30-changed-y',
    '30-original-x','30-original-y',
    '45-changed-x','45-changed-y',
    '45-original-x','45-original-y',
    '60-changed-x','60-changed-y',
    '60-original-x','60-original-y'
]
coords.to_csv(f"{out_dir}/unchainedCoordsMerge.csv",index=False)

In [5]:
kmeansResult=np.load("/p2/zulab/jtian/data/SA/05_CAST/dn3/output_cast1/demo1_kmeans_labels_k20.npz", allow_pickle=True)

In [6]:
kmeansResult

In [11]:
kmeansResult.files

['0', '15', '30', '45', '60']

In [13]:
cell_label_dict={}
for f,a in kmeansResult.items():
    cell_label_dict[f]=a

In [14]:
cell_label_dict

{'0': array([ 7,  7,  7, ..., 13, 13, 13], dtype=int32),
 '15': array([7, 7, 7, ..., 7, 7, 7], dtype=int32),
 '30': array([13,  7,  7, ...,  7,  7,  7], dtype=int32),
 '45': array([13, 13, 13, ...,  7,  7,  7], dtype=int32),
 '60': array([13, 13, 13, ..., 13, 13, 13], dtype=int32)}

In [18]:
for key, labels in cell_label_dict.items():

    labels_full = np.full(len(coords), np.nan)
    labels_full[:len(labels)] = labels

    col_name = f"{key}-original-x"
    idx = coords.columns.get_loc(col_name)

    coords.insert(idx, f"{key}-label", labels_full)

In [19]:
coords

,0-label,0-original-x,0-original-y,15-changed-x,15-changed-y,15-label,15-original-x,15-original-y,30-changed-x,30-changed-y,...,45-changed-x,45-changed-y,45-label,45-original-x,45-original-y,60-changed-x,60-changed-y,60-label,60-original-x,60-original-y
0,7.0,56099.0,6544.0,55889.605469,87.594566,7.0,43279.0,11864.0,43218.527344,11797.666992,...,29600.683594,11409.931641,13.0,15479.0,11404.0,15451.801758,11458.494141,13.0,2279.0,12064.0
1,7.0,56119.0,6544.0,55869.636719,89.182701,7.0,43299.0,11864.0,43238.296875,11798.726562,...,29620.697266,11409.502930,13.0,15499.0,11404.0,15472.122070,11459.142578,13.0,2299.0,12064.0
2,7.0,55959.0,6524.0,55849.664062,90.771080,7.0,43319.0,11864.0,43258.062500,11799.787109,...,29640.712891,11409.074219,13.0,15519.0,11404.0,15492.442383,11459.791016,13.0,2319.0,12064.0
3,7.0,55979.0,6524.0,55951.171875,103.617516,7.0,43219.0,11844.0,43277.832031,11800.847656,...,29660.726562,11408.645508,13.0,15539.0,11404.0,15512.762695,11460.439453,13.0,2339.0,12064.0
4,7.0,55999.0,6524.0,55931.199219,105.205650,7.0,43239.0,11844.0,43297.601562,11801.907227,...,29680.740234,11408.216797,13.0,15559.0,11404.0,15533.083008,11461.087891,13.0,2359.0,12064.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63277,13.0,56099.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63278,13.0,56119.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63279,13.0,56139.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63280,13.0,56159.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
coords.to_csv(f"{out_dir}/unchainedCoordsLabel.csv",index=False)

In [21]:
coords0  = coords.iloc[:, 0:3].copy()
coords15 = coords.iloc[:, 3:8].copy()
coords30 = coords.iloc[:, 8:13].copy()
coords45 = coords.iloc[:, 13:18].copy()
coords60 = coords.iloc[:, 18:23].copy()
coords0  = coords0.sort_values(by='0-label').reset_index(drop=False)
coords15 = coords15.sort_values(by='15-label').reset_index(drop=False)
coords30 = coords30.sort_values(by='30-label').reset_index(drop=False)
coords45 = coords45.sort_values(by='45-label').reset_index(drop=False)
coords60 = coords60.sort_values(by='60-label').reset_index(drop=False)

In [22]:
coords0

,index,0-label,0-original-x,0-original-y
0,38350,0.0,56579.0,2824.0
1,45816,0.0,56239.0,2204.0
2,45817,0.0,56259.0,2204.0
3,45818,0.0,56279.0,2204.0
4,45819,0.0,56299.0,2204.0
...,...,...,...,...
63277,13722,19.0,56779.0,4824.0
63278,13723,19.0,56799.0,4824.0
63279,33103,19.0,56799.0,3244.0
63280,16406,19.0,54899.0,4584.0


In [23]:
coords0  = coords0.rename(columns={'index': '0-index'})
coords15 = coords15.rename(columns={'index': '15-index'})
coords30 = coords30.rename(columns={'index': '30-index'})
coords45 = coords45.rename(columns={'index': '45-index'})
coords60 = coords60.rename(columns={'index': '60-index'})

In [24]:
samples = [0,15,30,45,60]
coords_dict = {
    0: coords0,
    15: coords15,
    30: coords30,
    45: coords45,
    60: coords60
}


for i in samples:
    df = coords_dict[i]
    print(df[f'{i}-label'].unique())

[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19.]
[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19. nan]
[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19. nan]
[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19. nan]
[ 0.  1.  2.  3.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17. 18.
 19. nan]


In [26]:
label_dfs = {}

for lab in range(20):   # 0–19
    lab = float(lab)    # 转成浮点数
    
    pieces = []
    
    for s, df in coords_dict.items():
        label_col = f'{s}-label'
        
        part = df[df[label_col] == lab].reset_index(drop=True)
        pieces.append(part)
    
    label_dfs[lab] = pd.concat(pieces, axis=1)  # 自动补NaN

In [27]:
label_dfs[0]

,0-index,0-label,0-original-x,0-original-y,15-index,15-changed-x,15-changed-y,15-label,15-original-x,15-original-y,...,45-changed-y,45-label,45-original-x,45-original-y,60-index,60-changed-x,60-changed-y,60-label,60-original-x,60-original-y
0,38350.0,0.0,56579.0,2824.0,41214.0,55840.808594,4275.317383,0.0,43659.0,7864.0,...,9330.250000,0.0,15379.0,9344.0,12583.0,15507.547852,9711.485352,0.0,2279.0,10384.0
1,45816.0,0.0,56239.0,2204.0,40969.0,55819.187500,4256.117676,0.0,43679.0,7884.0,...,9290.683594,0.0,15339.0,9304.0,35206.0,15264.460938,7767.571289,0.0,1979.0,8524.0
2,45817.0,0.0,56259.0,2204.0,21199.0,56187.851562,2574.180176,0.0,43179.0,9464.0,...,9291.541016,0.0,15299.0,9304.0,35207.0,15284.781250,7768.219727,0.0,1999.0,8524.0
3,45818.0,0.0,56279.0,2204.0,21200.0,56167.878906,2575.768555,0.0,43199.0,9464.0,...,9292.399414,0.0,15259.0,9304.0,37756.0,16470.003906,7597.850586,0.0,3159.0,8324.0
4,45819.0,0.0,56299.0,2204.0,21201.0,56147.910156,2577.356934,0.0,43219.0,9464.0,...,9292.828125,0.0,15239.0,9304.0,37755.0,16449.683594,7597.202148,0.0,3139.0,8324.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
for key, df in label_dfs.items():
    label_dfs[key] = df.loc[:, ~df.columns.str.contains('label')]

In [29]:
label_dfs[0]

,0-index,0-original-x,0-original-y,15-index,15-changed-x,15-changed-y,15-original-x,15-original-y,30-index,30-changed-x,...,45-index,45-changed-x,45-changed-y,45-original-x,45-original-y,60-index,60-changed-x,60-changed-y,60-original-x,60-original-y
0,38350.0,56579.0,2824.0,41214.0,55840.808594,4275.317383,43659.0,7864.0,39576,44368.554688,...,17697.0,29456.005859,9330.250000,15379.0,9344.0,12583.0,15507.547852,9711.485352,2279.0,10384.0
1,45816.0,56239.0,2204.0,40969.0,55819.187500,4256.117676,43679.0,7884.0,42277,44261.410156,...,18176.0,29415.111328,9290.683594,15339.0,9304.0,35206.0,15264.460938,7767.571289,1979.0,8524.0
2,45817.0,56259.0,2204.0,21199.0,56187.851562,2574.180176,43179.0,9464.0,42278,44281.179688,...,18174.0,29375.083984,9291.541016,15299.0,9304.0,35207.0,15284.781250,7768.219727,1999.0,8524.0
3,45818.0,56279.0,2204.0,21200.0,56167.878906,2575.768555,43199.0,9464.0,35465,43027.398438,...,18172.0,29335.054688,9292.399414,15259.0,9304.0,37756.0,16470.003906,7597.850586,3159.0,8324.0
4,45819.0,56299.0,2204.0,21201.0,56147.910156,2577.356934,43219.0,9464.0,42279,44300.945312,...,18171.0,29315.041016,9292.828125,15239.0,9304.0,37755.0,16449.683594,7597.202148,3139.0,8324.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24588,42903.511719,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36028,44037.671875,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35981,43108.554688,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17731,42953.402344,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
with open(f'{out_dir}/label_dfs.pkl', 'wb') as f:
    pickle.dump(label_dfs, f)

In [31]:
coords=pd.read_csv(f"{out_dir}/unchainedCoordsMerge.csv")

In [32]:
coords

,0-original-x,0-original-y,15-changed-x,15-changed-y,15-original-x,15-original-y,30-changed-x,30-changed-y,30-original-x,30-original-y,45-changed-x,45-changed-y,45-original-x,45-original-y,60-changed-x,60-changed-y,60-original-x,60-original-y
0,56099.0,6544.0,55889.605,87.594570,43279.0,11864.0,43218.527,11797.667,29659.0,11404.0,29600.684,11409.9320,15479.0,11404.0,15451.802,11458.494,2279.0,12064.0
1,56119.0,6544.0,55869.637,89.182700,43299.0,11864.0,43238.297,11798.727,29679.0,11404.0,29620.697,11409.5030,15499.0,11404.0,15472.122,11459.143,2299.0,12064.0
2,55959.0,6524.0,55849.664,90.771080,43319.0,11864.0,43258.062,11799.787,29699.0,11404.0,29640.713,11409.0740,15519.0,11404.0,15492.442,11459.791,2319.0,12064.0
3,55979.0,6524.0,55951.170,103.617516,43219.0,11844.0,43277.832,11800.848,29719.0,11404.0,29660.727,11408.6455,15539.0,11404.0,15512.763,11460.439,2339.0,12064.0
4,55999.0,6524.0,55931.200,105.205650,43239.0,11844.0,43297.600,11801.907,29739.0,11404.0,29680.740,11408.2170,15559.0,11404.0,15533.083,11461.088,2359.0,12064.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63277,56099.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63278,56119.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63279,56139.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63280,56159.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
def chain_match_one_df_hungarian(df: pd.DataFrame, samples=(0, 15, 30, 45, 60)) -> pd.DataFrame:
    """
    对单个 df 做链式匹配：
    起点 = sample 0 的 original 坐标有效的所有行
    然后依次在 15 / 30 / 45 / 60 的 changed 坐标中，
    用匈牙利算法做“当前所有活跃链”的全局一对一最优匹配。

    返回：
        out_df：与原 df 列名一致的新 DataFrame
                每一行代表一条从 sample 0 出发的链
    """

    df = df.copy()

    def c(s, name):
        return f"{s}-{name}"

    # ---------- 1. 找到 sample 0 的起点 ----------
    o0x, o0y = c(0, "original-x"), c(0, "original-y")
    i0col = c(0, "index")

    start_mask = df[o0x].notna() & df[o0y].notna()
    start_indices = df.index[start_mask].to_numpy()

    if len(start_indices) == 0:
        return pd.DataFrame(columns=df.columns)

    # 输出表：先全部填 NA，后面逐步写入
    out_df = pd.DataFrame(pd.NA, index=start_indices, columns=df.columns)

    # 先把 sample 0 的信息写进去
    if i0col in df.columns:
        out_df.loc[start_indices, i0col] = df.loc[start_indices, i0col].to_numpy()
    out_df.loc[start_indices, o0x] = df.loc[start_indices, o0x].to_numpy()
    out_df.loc[start_indices, o0y] = df.loc[start_indices, o0y].to_numpy()

    # 当前“还在继续往后匹配”的链
    # active_out_idx: out_df 中哪些行目前还是活跃的
    # q: 这些活跃链当前要去匹配的查询点坐标
    active_out_idx = start_indices.copy()
    q = df.loc[start_indices, [o0x, o0y]].to_numpy(dtype=float, copy=True)

    # ---------- 2. 依次做 15 / 30 / 45 / 60 ----------
    for s in samples[1:]:
        if len(active_out_idx) == 0:
            break

        idx_col = c(s, "index")
        cx, cy = c(s, "changed-x"), c(s, "changed-y")
        ox, oy = c(s, "original-x"), c(s, "original-y")

        # 该样本没有 changed 坐标列，就断掉
        if (cx not in df.columns) or (cy not in df.columns):
            break

        # 候选池：该样本中所有 changed 坐标有效的行
        cand_mask = df[cx].notna() & df[cy].notna()
        cand_rows = df.index[cand_mask].to_numpy()

        if len(cand_rows) == 0:
            break

        cand_xy = df.loc[cand_rows, [cx, cy]].to_numpy(dtype=float, copy=True)

        # ---------- 3. 代价矩阵 ----------
        # cost[i, j] = 活跃链 i 的当前查询点 q[i] 到 候选点 j 的 changed 坐标 的平方欧氏距离
        # 用平方距离即可，不影响最优匹配结果，还省一次开方
        cost = cdist(q, cand_xy, metric="sqeuclidean")

        # ---------- 4. 匈牙利算法：求当前这一步的全局最优一对一匹配 ----------
        # row_ind 对应 q 的行号
        # col_ind 对应 cand_xy 的列号
        row_ind, col_ind = linear_sum_assignment(cost)

        # 被成功匹配到的 out_df 行
        matched_out_idx = active_out_idx[row_ind]

        # 被选中的 df 候选行
        matched_df_rows = cand_rows[col_ind]

        # ---------- 5. 把匹配结果写回输出表 ----------
        if idx_col in df.columns:
            out_df.loc[matched_out_idx, idx_col] = df.loc[matched_df_rows, idx_col].to_numpy()
        if cx in df.columns:
            out_df.loc[matched_out_idx, cx] = df.loc[matched_df_rows, cx].to_numpy()
        if cy in df.columns:
            out_df.loc[matched_out_idx, cy] = df.loc[matched_df_rows, cy].to_numpy()
        if ox in df.columns:
            out_df.loc[matched_out_idx, ox] = df.loc[matched_df_rows, ox].to_numpy()
        if oy in df.columns:
            out_df.loc[matched_out_idx, oy] = df.loc[matched_df_rows, oy].to_numpy()

        # ---------- 6. 为下一步准备新的查询点 ----------
        # 下一步查询点用这一步匹配到的 original 坐标
        # 只有 original-x / original-y 都不缺失的，才能继续往后链接
        next_valid_mask = (
            df.loc[matched_df_rows, [ox, oy]].notna().all(axis=1).to_numpy()
            if (ox in df.columns and oy in df.columns)
            else np.zeros(len(matched_df_rows), dtype=bool)
        )

        active_out_idx = matched_out_idx[next_valid_mask]
        next_df_rows = matched_df_rows[next_valid_mask]

        if len(next_df_rows) == 0:
            break

        q = df.loc[next_df_rows, [ox, oy]].to_numpy(dtype=float, copy=True)

    # 如果你想把输出索引改成 0,1,2,... 可以取消下一行注释
    # out_df = out_df.reset_index(drop=True)

    return out_df

In [34]:
matched_label_dfs = {
    label: chain_match_one_df_hungarian(df)
    for label, df in label_dfs.items()
}

In [35]:

with open(f'{out_dir}/matched_label_dfs_all.pkl', "wb") as f:
    pickle.dump(matched_label_dfs, f)

In [36]:
for key, df in matched_label_dfs.items():
    matched_label_dfs[key] = df.loc[:, df.columns.str.contains('original')]

In [37]:
matched_label_dfs[0]

,0-original-x,0-original-y,15-original-x,15-original-y,30-original-x,30-original-y,45-original-x,45-original-y,60-original-x,60-original-y
0,56579.0,2824.0,42819.0,9184.0,29119.0,8744.0,15019.0,8764.0,1699.0,9584.0
1,56239.0,2204.0,43119.0,9824.0,29419.0,9344.0,15339.0,9364.0,2139.0,10104.0
2,56259.0,2204.0,43099.0,9844.0,29399.0,9384.0,15339.0,9404.0,2159.0,10164.0
3,56279.0,2204.0,43079.0,9844.0,29379.0,9384.0,15299.0,9384.0,2099.0,10104.0
4,56299.0,2204.0,43059.0,9844.0,29359.0,9384.0,15279.0,9384.0,2079.0,10084.0
...,...,...,...,...,...,...,...,...,...,...
1836,55259.0,2824.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1837,55259.0,2884.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1838,55899.0,2924.0,43379.0,9184.0,29719.0,8704.0,15539.0,8684.0,2139.0,9564.0
1839,55279.0,2884.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [38]:
with open(f'{out_dir}/matched_label_dfs_ori.pkl', "wb") as f:
    pickle.dump(matched_label_dfs, f)

In [39]:
dfs = []

for label, df in matched_label_dfs.items():
    df_copy = df.copy()
    df_copy.insert(0, 'cell-label', label) 
    dfs.append(df_copy)
chainedOCoordsLabel = pd.concat(dfs, ignore_index=True)

In [40]:
chainedOCoordsLabel

,cell-label,0-original-x,0-original-y,15-original-x,15-original-y,30-original-x,30-original-y,45-original-x,45-original-y,60-original-x,60-original-y
0,0.0,56579.0,2824.0,42819.0,9184.0,29119.0,8744.0,15019.0,8764.0,1699.0,9584.0
1,0.0,56239.0,2204.0,43119.0,9824.0,29419.0,9344.0,15339.0,9364.0,2139.0,10104.0
2,0.0,56259.0,2204.0,43099.0,9844.0,29399.0,9384.0,15339.0,9404.0,2159.0,10164.0
3,0.0,56279.0,2204.0,43079.0,9844.0,29379.0,9384.0,15299.0,9384.0,2099.0,10104.0
4,0.0,56299.0,2204.0,43059.0,9844.0,29359.0,9384.0,15279.0,9384.0,2079.0,10084.0
...,...,...,...,...,...,...,...,...,...,...,...
63277,19.0,56779.0,4824.0,42779.0,7364.0,28959.0,6904.0,14979.0,6944.0,1739.0,7784.0
63278,19.0,56799.0,4824.0,42779.0,7324.0,28939.0,6864.0,14959.0,6904.0,1719.0,7744.0
63279,19.0,56799.0,3244.0,42579.0,8684.0,28819.0,8264.0,14759.0,8304.0,1479.0,9044.0
63280,19.0,54899.0,4584.0,44499.0,7604.0,30659.0,6964.0,16599.0,7064.0,3219.0,7884.0


In [41]:
chainedOCoordsLabel = chainedOCoordsLabel.dropna()

In [43]:
chainedOCoordsLabel.to_csv(f"{out_dir}/chainedOCoordsLabel.csv",index=False)

In [44]:
obs = adata.obs[['x', 'y']].copy()
obs = obs.reset_index().rename(columns={'SpotIndex': 'obs_index'})

obs['x'] = obs['x'].astype('Int64')
obs['y'] = obs['y'].astype('Int64')

lookup = obs.set_index(['x', 'y'])['obs_index']

samples = [0, 15, 30, 45, 60]

for s in samples:
    sx = f'{s}-original-x'
    sy = f'{s}-original-y'
    out_col = f'{s}-index'

    xk = chainedOCoordsLabel[sx].astype('Int64')
    yk = chainedOCoordsLabel[sy].astype('Int64')

    mi = pd.MultiIndex.from_arrays([xk, yk], names=['x', 'y'])
    chainedOCoordsLabel[out_col] = lookup.reindex(mi).to_numpy()

In [45]:
chainedOCoordsLabel

,cell-label,0-original-x,0-original-y,15-original-x,15-original-y,30-original-x,30-original-y,45-original-x,45-original-y,60-original-x,60-original-y,0-index,15-index,30-index,45-index,60-index
0,0.0,56579.0,2824.0,42819.0,9184.0,29119.0,8744.0,15019.0,8764.0,1699.0,9584.0,38351,87905,148202,210682,269620
1,0.0,56239.0,2204.0,43119.0,9824.0,29419.0,9344.0,15339.0,9364.0,2139.0,10104.0,45817,80168,140891,203286,263423
2,0.0,56259.0,2204.0,43099.0,9844.0,29399.0,9384.0,15339.0,9404.0,2159.0,10164.0,45818,79931,140417,202809,262728
3,0.0,56279.0,2204.0,43079.0,9844.0,29379.0,9384.0,15299.0,9384.0,2099.0,10104.0,45819,79930,140416,203045,263421
4,0.0,56299.0,2204.0,43059.0,9844.0,29359.0,9384.0,15279.0,9384.0,2079.0,10084.0,45820,79929,140415,203044,263653
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63277,19.0,56779.0,4824.0,42779.0,7364.0,28959.0,6904.0,14979.0,6944.0,1739.0,7784.0,13723,110387,171065,233411,291832
63278,19.0,56799.0,4824.0,42779.0,7324.0,28939.0,6864.0,14959.0,6904.0,1719.0,7744.0,13724,110842,171525,233876,292296
63279,19.0,56799.0,3244.0,42579.0,8684.0,28819.0,8264.0,14759.0,8304.0,1479.0,9044.0,33104,94166,154201,216476,276301
63280,19.0,54899.0,4584.0,44499.0,7604.0,30659.0,6964.0,16599.0,7064.0,3219.0,7884.0,16407,107676,170452,232077,290735


In [46]:
chainedIndexLabel = chainedOCoordsLabel[['cell-label','0-index','15-index','30-index','45-index','60-index']]

In [47]:
chainedIndexLabel

,cell-label,0-index,15-index,30-index,45-index,60-index
0,0.0,38351,87905,148202,210682,269620
1,0.0,45817,80168,140891,203286,263423
2,0.0,45818,79931,140417,202809,262728
3,0.0,45819,79930,140416,203045,263421
4,0.0,45820,79929,140415,203044,263653
...,...,...,...,...,...,...
63277,19.0,13723,110387,171065,233411,291832
63278,19.0,13724,110842,171525,233876,292296
63279,19.0,33104,94166,154201,216476,276301
63280,19.0,16407,107676,170452,232077,290735


In [ ]:
chainedIndexLabel = chainedIndexLabel.astype(int)

In [48]:
chainedIndexLabel.to_csv(f"{out_dir}/chainedIndexLabel.csv",index=False)